# Checkpoint F1 Visualization

Muc tieu cua notebook nay:

1. Nap lai cac checkpoint da luu cua E1-E4.
2. Danh gia lai tren validation split bang cung evaluator cua project.
3. Ve bieu do F1 de lam bang chung chon model tot nhat.

Luu y phuong phap: validation dung de chon winner. Official test chi dung sau khi winner da khoa.

In [ ]:
from pathlib import Path
import gc
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

from src.data.dataset import load_examples
from src.evaluation.evaluator import evaluate
from src.evaluation.loaders import build_eval_dataloader
from src.models.factory import load_model_from_checkpoint
from src.utils.io import read_json, write_json

pd.set_option("display.max_columns", 50)
PROJECT_ROOT

## Cau hinh

Mac dinh notebook danh gia ca 4 checkpoint tren full validation split. Neu chi muon test nhanh notebook co chay khong, dat `LIMIT_EXAMPLES = 200`; khi lam bao cao thi de `None`.

In [ ]:
EXPERIMENT_IDS = ["E1", "E2", "E3", "E4"]
SPLIT = "validation"
LIMIT_EXAMPLES = None

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_cuda = device.type == "cuda"

print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {device}")
if use_cuda:
    print(torch.cuda.get_device_name(0))

## Load validation data

In [ ]:
split_path = PROJECT_ROOT / "data" / "processed" / f"{SPLIT}.jsonl"
examples = load_examples(split_path, limit=LIMIT_EXAMPLES)

num_examples = len(examples)
num_words = sum(len(ex.tokens) for ex in examples)
print(f"Loaded {num_examples:,} examples / {num_words:,} words from {split_path}")

## Evaluate saved checkpoints

Ham duoi day nap model tu checkpoint folder, tao dataloader dung tokenizer/vocabulary da luu trong checkpoint, roi tinh lai metric bang `evaluate()`.

In [ ]:
def get_weight_mode(meta):
    cfg = meta.get("training_config", {})
    loss_cfg = cfg.get("loss", {}) if isinstance(cfg, dict) else {}
    return loss_cfg.get("weight_mode", "unknown")


def evaluate_checkpoint(exp_id, examples):
    ckpt_dir = PROJECT_ROOT / "outputs" / "checkpoints" / exp_id
    print(f"\n=== {exp_id}: loading {ckpt_dir} ===")

    model, meta = load_model_from_checkpoint(ckpt_dir, device=device)
    model_type = meta["model_type"]
    batch_size = 8 if model_type == "phobert" else 256

    loader, _, _ = build_eval_dataloader(
        ckpt_dir,
        examples,
        batch_size=batch_size,
        num_workers=0,
    )

    metrics = evaluate(
        model,
        loader,
        device=device,
        use_amp=(use_cuda and model_type == "phobert"),
        progress=True,
        desc=f"{exp_id} {SPLIT}",
    )

    saved_best = meta.get("best_score")
    recomputed = metrics["punctuation_macro_f1"]
    diff = None if saved_best is None else recomputed - float(saved_best)

    row = {
        "experiment": exp_id,
        "model": meta.get("model_name", model_type),
        "model_type": model_type,
        "weight_mode": get_weight_mode(meta),
        "best_epoch": meta.get("best_epoch"),
        "saved_best_punctuation_macro_f1": saved_best,
        "recomputed_punctuation_macro_f1": recomputed,
        "recomputed_minus_saved": diff,
        "accuracy": metrics["accuracy"],
        "macro_f1_all_4_classes": metrics["macro_f1"],
        "f1_O": metrics["per_class"]["O"]["f1"],
        "f1_COMMA": metrics["per_class"]["COMMA"]["f1"],
        "f1_PERIOD": metrics["per_class"]["PERIOD"]["f1"],
        "f1_QUESTION": metrics["per_class"]["QUESTION"]["f1"],
        "num_evaluated_tokens": metrics["num_evaluated_tokens"],
    }

    del model, loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return row, metrics


In [ ]:
rows = []
metrics_by_experiment = {}

for exp_id in EXPERIMENT_IDS:
    row, metrics = evaluate_checkpoint(exp_id, examples)
    rows.append(row)
    metrics_by_experiment[exp_id] = metrics

df = pd.DataFrame(rows).sort_values("recomputed_punctuation_macro_f1", ascending=False)
df

## Save recomputed metrics

In [ ]:
eval_dir = PROJECT_ROOT / "outputs" / "evaluation"
fig_dir = PROJECT_ROOT / "outputs" / "figures"
eval_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)

csv_path = eval_dir / "checkpoint_validation_f1_recomputed.csv"
json_path = eval_dir / "checkpoint_validation_f1_recomputed.json"

df.to_csv(csv_path, index=False, encoding="utf-8-sig")
write_json(
    json_path,
    {
        "split": SPLIT,
        "limit_examples": LIMIT_EXAMPLES,
        "selection_metric": "punctuation_macro_f1",
        "note": "Metrics were recomputed by loading saved checkpoints and evaluating on the validation split.",
        "ranking": df.to_dict(orient="records"),
    },
)

print(csv_path)
print(json_path)

## Plot 1: headline validation Punctuation Macro-F1

In [ ]:
plot_df = df.sort_values("recomputed_punctuation_macro_f1", ascending=True).copy()
winner = df.iloc[0]["experiment"]

colors = ["#9ca3af" if exp != winner else "#2563eb" for exp in plot_df["experiment"]]

fig, ax = plt.subplots(figsize=(8.5, 4.8))
bars = ax.barh(plot_df["experiment"], plot_df["recomputed_punctuation_macro_f1"], color=colors)

ax.set_title("Validation Punctuation Macro-F1 from saved checkpoints")
ax.set_xlabel("Punctuation Macro-F1 (COMMA / PERIOD / QUESTION)")
ax.set_ylabel("Experiment")
ax.set_xlim(0, 1.0)
ax.grid(axis="x", alpha=0.25)

for bar, value in zip(bars, plot_df["recomputed_punctuation_macro_f1"]):
    ax.text(value + 0.01, bar.get_y() + bar.get_height() / 2, f"{value:.4f}", va="center")

fig.tight_layout()
fig_path = fig_dir / "checkpoint_validation_f1_recomputed.png"
fig.savefig(fig_path, dpi=200, bbox_inches="tight")
plt.show()

fig_path

## Plot 2: per-class punctuation F1

In [ ]:
per_class_df = df.set_index("experiment")[["f1_COMMA", "f1_PERIOD", "f1_QUESTION"]]
per_class_df = per_class_df.rename(columns={
    "f1_COMMA": "COMMA",
    "f1_PERIOD": "PERIOD",
    "f1_QUESTION": "QUESTION",
})

ax = per_class_df.plot(kind="bar", figsize=(8.5, 4.8), rot=0)
ax.set_title("Validation per-class F1 from saved checkpoints")
ax.set_xlabel("Experiment")
ax.set_ylabel("F1")
ax.set_ylim(0, 1.0)
ax.grid(axis="y", alpha=0.25)
ax.legend(title="Label")

plt.tight_layout()
fig_path = fig_dir / "checkpoint_validation_per_class_f1_recomputed.png"
plt.savefig(fig_path, dpi=200, bbox_inches="tight")
plt.show()

fig_path

## Report sentence

In [ ]:
best = df.iloc[0]
runner_up = df.iloc[1]
margin = best["recomputed_punctuation_macro_f1"] - runner_up["recomputed_punctuation_macro_f1"]

print(
    f"The saved checkpoints were reloaded and evaluated on the validation split. "
    f"{best['experiment']} achieved the highest Punctuation Macro-F1 "
    f"({best['recomputed_punctuation_macro_f1']:.4f}), ahead of "
    f"{runner_up['experiment']} by {margin:.4f}. Therefore {best['experiment']} "
    f"is selected as the best model before any official test evaluation."
)

## Optional: official test result of the locked winner

Cell nay khong chon model. No chi doc artifact test da tao sau khi winner da khoa.

In [ ]:
final_report_path = PROJECT_ROOT / "outputs" / "evaluation" / "final_report.json"
if final_report_path.exists():
    final_report = read_json(final_report_path)
    official_test = final_report["official_test"]
    print("Official test of locked winner")
    print(f"Punctuation Macro-F1: {official_test['punctuation_macro_f1']:.4f}")
    print(f"Accuracy: {official_test['accuracy']:.4f}")
    print(f"Macro-F1 all 4 classes: {official_test['macro_f1']:.4f}")
else:
    print(f"Not found: {final_report_path}")